# Active-set QP: `method="active-set"` / `solver_selection="qp-active-set"`

POUNCE ships two solvers for convex LPs and QPs:

* the **convex interior-point** solver (`pounce-convex`) — the default, and what
  `auto` picks;
* the **parametric active-set** engine (`pounce-qp`) — opt-in.

This notebook shows how to reach the active-set engine, what it is good at,
and — with numbers — where it is clearly the wrong choice. If you only read one
line: **for a cold one-shot convex QP, prefer the interior-point solver.**
The active-set engine earns its place on small-to-medium dense problems and on
*sequences* of related QPs.

In [1]:
import time

import numpy as np

import pounce
from pounce.qp import solve_qp

np.set_printoptions(precision=6, suppress=True)

## 1. One API, two engines

`solve_qp(..., method=...)` selects the engine. Both minimize
`½xᵀPx + cᵀx` subject to `Ax = b`, `Gx ≤ h`, `lb ≤ x ≤ ub`.

Take `min (x₀−3)² + (x₁−2)²` subject to `x₀ + x₁ ≤ 4`, `x ≥ 0`. The
unconstrained optimum `(3, 2)` violates the row, so it binds and the answer is
the projection onto it: `x* = (2.5, 1.5)`.

In [2]:
P = np.diag([2.0, 2.0])
c = np.array([-6.0, -4.0])          # (x-3)^2 + (x-2)^2, constant 13 dropped
G = np.array([[1.0, 1.0]])
h = np.array([4.0])
lb = np.zeros(2)

for method in ('ipm', 'active-set'):
    r = solve_qp(P=P, c=c, G=G, h=h, lb=lb, method=method)
    print(f'{method:11s} status={r.status:8s} x={r.x}  obj={r.obj:.9f}')

ipm         status=optimal  x=[2.5 1.5]  obj=-12.499999999
active-set  status=optimal  x=[2.5 1.5]  obj=-12.500000000


Same point, same objective. The engines differ in *how* they get there, not in
what the answer is.

## 2. From `minimize`, and why both surfaces now agree

`solver_selection="qp-active-set"` reaches the same engine from `minimize`,
and — as of this release — from the CLI too. That was not always true: the
selector used to name the convex active-set driver on the CLI and the *SQP
outer loop* in `minimize`, so the same string meant two different algorithms
depending on how you called POUNCE.

The tell for which engine ran is `nfev`. The convex route consumes the
extracted quadratic form and never calls back into Python, so a routed solve
reports **zero** function evaluations; the NLP path reports many.

In [3]:
fun = lambda x: (x[0] - 3.0) ** 2 + (x[1] - 2.0) ** 2
jac = lambda x: np.array([2 * (x[0] - 3.0), 2 * (x[1] - 2.0)])
con = [{'type': 'ineq',
        'fun': lambda x: np.array([4.0 - x[0] - x[1]]),
        'jac': lambda x: np.array([[-1.0, -1.0]])}]
kw = dict(jac=jac, constraints=con, bounds=[(0, None), (0, None)])

for sel in ('qp-ipm', 'qp-active-set', 'nlp'):
    r = pounce.minimize(fun, np.zeros(2), solver_selection=sel, **kw)
    print(f'{sel:15s} x={r.x}  fun={r.fun:.9f}  nfev={r.nfev}')

qp-ipm          x=[2.5 1.5]  fun=0.500000001  nfev=0
qp-active-set   x=[2.5 1.5]  fun=0.500000000  nfev=0
nlp             x=[2.5 1.5]  fun=0.499999993  nfev=9


`nfev=0` for both convex selectors confirms they routed to the convex solvers;
the NLP path evaluates the callbacks.

## 3. Where the active-set engine wins

On small-to-medium **dense**, well-conditioned QPs it does less work per solve.
An interior-point solve costs a fixed-ish number of expensive factorizations;
an active-set solve costs one cheap update per active-set change, and on a
well-behaved problem there are few.

In [4]:
rng = np.random.default_rng(0)
n, m = 60, 30
A = rng.standard_normal((n, n))
P_big = A @ A.T + 0.5 * np.eye(n)          # dense SPD
c_big = rng.standard_normal(n)
G_big = rng.standard_normal((m, n))
h_big = rng.standard_normal(m) + 2.0

for method in ('ipm', 'active-set'):
    t0 = time.perf_counter()
    for _ in range(20):
        r = solve_qp(P=P_big, c=c_big, G=G_big, h=h_big, method=method)
    dt = (time.perf_counter() - t0) / 20
    print(f'{method:11s} obj={r.obj:.9f}  {dt * 1e3:6.2f} ms/solve')

ipm         obj=-5.442057431    4.84 ms/solve
active-set  obj=-5.442057431    2.63 ms/solve


Same objective to nine digits, in roughly half the time. Your mileage depends
on conditioning and on how many constraints are active at the optimum.

## 4. Where it loses — and it loses badly

The honest benchmark. All 138 Maros-Mészáros convex QPs, 120 s cap, graded
against published optima:

| engine | correct |
|---|---|
| interior-point (`method="ipm"`, and what `auto` picks) | **137 / 138** |
| active-set | 71 / 138 |

That gap is not a bug list, it is the character of the method. An active-set
iteration count is *combinatorial* in the size of the active set, so large,
degenerate problems exhaust the iteration budget; an interior-point count is
nearly independent of problem size. This is why `auto` never selects the
active-set engine, and why you should not reach for it on an unfamiliar large
problem.

**What it will not do is lie.** Across all 138 problems and every configuration
tested, it never returned a successful status with a wrong objective: every
claimed optimum is re-verified against the original problem's KKT conditions
before it is reported, and a solve that cannot be verified is downgraded to
`Maximum_Iterations_Exceeded` or a numerical failure. A failure is visible, not
silent.

## 5. Limitations to know before you opt in

1. **Cold one-shot large/degenerate QPs** — use the IPM (section 4).
2. **`warm_start=` is not supported** with `method="active-set"`. The engine
   warm-starts from a *working set*, not a primal-dual point, so accepting one
   would be misleading. It raises rather than silently ignoring it.
3. **True parametric warm starting is Rust-only today.** `solve_parametric`
   traces the homotopy from a previous QP and its solution to a new one — the
   case this engine exists for (MPC steps, branch-and-bound nodes,
   continuation). It is not yet exposed to Python.
4. **A non-convex-QP problem is refused**, matching the CLI. For the active-set
   *SQP outer loop* on a general NLP, use `algorithm="active-set-sqp"`.
5. Some large instances that the IPM solves will hit the time cap here
   (`AUG2D`, `CONT-050`, `DTOC3`, …). Pass `sqp_qp_use_homotopy=no` on the CLI
   to fall back to the older conventional path if that suits your workload.

In [5]:
# (2) warm_start is rejected, not ignored
warm = solve_qp(P=P, c=c, G=G, h=h, lb=lb, method='ipm')
try:
    solve_qp(P=P, c=c, G=G, h=h, lb=lb, method='active-set', warm_start=warm)
except ValueError as e:
    print('warm_start :', str(e).split('(')[0].strip())

# (4) a non-convex-QP problem is refused rather than silently rerouted
try:
    pounce.minimize(lambda x: np.sin(x[0]) + x[1] ** 4, np.array([0.5, 0.5]),
                    constraints=[{'type': 'ineq',
                                  'fun': lambda x: np.array([1.0 - x[0]]),
                                  'jac': lambda x: np.array([[-1.0, 0.0]])}],
                    solver_selection='qp-active-set')
except ValueError as e:
    print('non-convex :', str(e)[:96], '...')

warm_start : solve_qp: warm_start= is not supported with method='active-set'
non-convex : solver_selection='qp-active-set' but the problem was not detected as a convex LP/QP (convex-quad ...


## Where next

* [`15_convex_qp.ipynb`](15_convex_qp.ipynb) — the interior-point solver in
  depth: duals, infeasibility and unboundedness certificates, warm starts,
  batches, factorization reuse.
* [`11_batched_warm_start.ipynb`](11_batched_warm_start.ipynb) — warm-started
  batches on the IPM path, which *is* exposed to Python.
* [`06_sqp_parametric_continuation.ipynb`](06_sqp_parametric_continuation.ipynb)
  — parametric continuation on the NLP side.
* `docs/src/lp-qp-routing.md` — how `solver_selection` resolves, and the exact
  contract each selector promises.